# Tools


Tools are a way to **extend an LLM's capabilities** by giving it **predefined functions** it can call. The model decides on its own whether a tool should be used to answer a query.


### How Tools Work (Step-by-Step)

1. **The model receives a query** along with a list of available tools.

2. **The model decides whether to use a tool** and generates an `AIMessage`.
   - If the message’s `finish_reason` is `tool_calls`, the model wants to call a tool.
   - The message includes the **function name** and **arguments**.

3. **Your server executes the tool** using the arguments the model provided.
   - You return the result as a `ToolMessage`.
   - The message includes a `tool_call_id` that links the result to the correct tool invocation.

4. **The model generates the final answer**, using both the original user query and the tool result.



### Example Flow

1. **Query:**  
   `"What is the current weather in London?"`  
   Available tools: `[get_current_weather]`

2. **Model decides:**  
   *"I want to call `get_current_weather` with argument `London`."*

3. **We run the tool:**  
   `get_current_weather("London") → "cloudy"`  
   Then we send the result back as a `ToolMessage`.

4. **Model replies:**  
   *"It is currently cloudy in London."*

Now that we understand the flow, let's recreate it using **LangChain**.

> **Note:** LangChain abstracts away many of the steps described above, so it may look like the model executes the tools directly.  
> But remember: **tools always run on your server**, never inside the model.


> ⚠️ **Environment Variables**  
> This notebook requires an <code>OPENAI_API_KEY</code> and <code>OPENWEATHER_API_KEY</code> to run.  

### Defining Tools

To create a tool in LangChain:

1. Write a normal Python function.  
2. Decorate it with `@tool`.  
3. Add a **docstring** describing:
   - what the tool does  
   - what arguments it accepts  
   - what it returns  

Docstrings are crucial because they provide the LLM with context so it can decide whether a tool is appropriate.

For more complex inputs, you can use **Pydantic models**, which allow structured arguments and metadata.

In [46]:
import requests
from langchain.tools import tool
from dotenv import load_dotenv
import os
from pprint import pprint
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage

load_dotenv()

True

We will use the **OpenWeatherMap API** to get the weather for a city.  
You can sign up for a free API key.

First, we need a **helper function** to convert a city name into coordinates using the geocoding API.  

This helper is *not* a tool — it is used *inside* the tool.

In [ ]:
API_KEY = os.getenv("OPENWEATHER_API_KEY")

def get_coords(city_name: str):
    URL = f"http://api.openweathermap.org/geo/1.0/direct?q={city_name}&appid={API_KEY}"
    response = requests.get(URL)

    if not response.ok:
        print(response.text)
        raise Exception("Something went wrong getting coordinates...")
    
    data = response.json()[0]
    lat, lon = data['lat'], data['lon']
    return lat, lon

get_coords('london')

(51.5073219, -0.1276474)

Next, we create the `get_current_weather` tool using the `@tool` decorator.  
The docstring describes the tool’s purpose, arguments, and return value.  
This helps the LLM know when and if it should call the tool.

In [63]:
@tool
def get_current_weather(city: str):
    """
    Gets current weather for a given city

    Args:
        city (str): The name of the city to get weather for, e.g., 'London'
    """

    lat, lon = get_coords(city)

    URL = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}"

    response = requests.get(URL)

    if not response.ok:
        print(response.text)
        raise Exception("Something went wrong getting weather...")
    
    return response.json()

#### Invoking the Tool Directly

A tool cannot be called like a regular Python function.  
After using the `@tool` decorator, it becomes a `StructuredTool`, which must be called using `.invoke()`.


In [106]:
print(type(get_current_weather))

<class 'langchain_core.tools.structured.StructuredTool'>


In [107]:
weather_data = get_current_weather.invoke('london')
pprint(weather_data)

{'base': 'stations',
 'clouds': {'all': 100},
 'cod': 200,
 'coord': {'lat': 51.5085, 'lon': -0.1257},
 'dt': 1763244066,
 'id': 2643743,
 'main': {'feels_like': 283.6,
          'grnd_level': 1005,
          'humidity': 92,
          'pressure': 1009,
          'sea_level': 1009,
          'temp': 284.05,
          'temp_max': 284.92,
          'temp_min': 283.46},
 'name': 'London',
 'sys': {'country': 'GB',
         'id': 268730,
         'sunrise': 1763191125,
         'sunset': 1763223102,
         'type': 2},
 'timezone': 0,
 'visibility': 10000,
 'weather': [{'description': 'overcast clouds',
              'icon': '04n',
              'id': 804,
              'main': 'Clouds'}],
 'wind': {'deg': 40, 'speed': 3.6}}


Now that we have our tool, we can create an agent using `create_agent`, passing it the list of tools the agent is allowed to use.


In [108]:
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.1,
)

agent = create_agent(
    model=llm,
    tools=[get_current_weather]
)

Once the agent is created, we can invoke it with a prompt and it will automatically handle tool selection and execution.

In [109]:
prompt = [
    SystemMessage(content='You are a helpful assistant. Use tools to produce the best and most accurate response.'),
    HumanMessage(content='What is the current weather in London?')
]

response = agent.invoke({'messages': prompt})
messages = [message.model_dump() for message in response['messages']]

## Understanding the Agent Response

Let's look at the response. We received 5 messages in total. 
The first two are the messages from our prompt.

In [110]:
for message in messages[:2]:
    print(f"{message['type'].upper()}: {message['content']}")

SYSTEM: You are a helpful assistant. Use tools to produce the best and most accurate response.
HUMAN: What is the current weather in London?


#### Tool Call

The third message is the interesting one. What we got: 
- AI message with `finish_reason` being `tool_calls`. This tells us that the model decided to use a tool by understanding our query and the tool we provided. 
- If the tool is called, `content` is always empty. 
- A list of `tool_calls` that the model wants to execute. As we can see, it wants to run `get_current_weather` with the argument `London`. 
- It also provided the `id` of the tool call. 


If we were calling the OpenAI API manually, we would now:

- run the tool function ourselves  
- send back a `ToolMessage` with the result and the same `tool_call_id`

But **LangChain does this for us automatically**.


In [112]:
tool_call = messages[2]
print(f"TYPE: {tool_call['type'].upper()}")
print(f"CONTENT: {tool_call['content']}")
print(f"FINISH_REASON: {tool_call['response_metadata']['finish_reason']}")
print(f"TOOL_CALLS: {tool_call['tool_calls']}")

TYPE: AI
CONTENT: 
FINISH_REASON: tool_calls
TOOL_CALLS: [{'name': 'get_current_weather', 'args': {'city': 'London'}, 'id': 'call_d45BXdnBKqsqHl7uYCbfy4TH', 'type': 'tool_call'}]


#### The ToolMessage
Let's look at the next message. This is a `ToolMessage` that is sent to the LLM after the tool is executed. We can see the type is `tool`, the `content` is the result of the tool (weather data), and we also included `tool_call_id`. This id matches the one we received from the LLM in the previous message ☝️. This lets the model know which tool invocation the result belongs to.

In [118]:
tool_message = messages[3]
print(f"TYPE: {tool_message['type']}")
print(f"TOOL_CALL_ID: {tool_message['tool_call_id']}")
print(f"NAME: {tool_message['name']}")
pprint(f"CONTENT: {tool_message['content']}")

TYPE: tool
TOOL_CALL_ID: call_d45BXdnBKqsqHl7uYCbfy4TH
NAME: get_current_weather
('CONTENT: {"coord": {"lon": -0.1257, "lat": 51.5085}, "weather": [{"id": 804, '
 '"main": "Clouds", "description": "overcast clouds", "icon": "04n"}], "base": '
 '"stations", "main": {"temp": 284.05, "feels_like": 283.6, "temp_min": '
 '283.46, "temp_max": 284.92, "pressure": 1009, "humidity": 92, "sea_level": '
 '1009, "grnd_level": 1005}, "visibility": 10000, "wind": {"speed": 3.6, '
 '"deg": 40}, "clouds": {"all": 100}, "dt": 1763244066, "sys": {"type": 2, '
 '"id": 268730, "country": "GB", "sunrise": 1763191125, "sunset": 1763223102}, '
 '"timezone": 0, "id": 2643743, "name": "London", "cod": 200}')


#### Final Model Response

Once the model receives the tool result, it uses that data to answer the original user query.  
The final message is an `AIMessage` containing the natural-language response generated by the LLM.

In [123]:
final_response = messages[-1]
print(f"TYPE: {final_response['type'].upper()}")
print(f"CONTENT: {final_response['content']}")

TYPE: AI
CONTENT: The current weather in London is as follows:

- **Condition**: Overcast clouds
- **Temperature**: 284.05 K (approximately 11.9 °C)
- **Feels Like**: 283.6 K (approximately 10.5 °C)
- **Humidity**: 92%
- **Wind Speed**: 3.6 m/s from the northeast
- **Visibility**: 10 km

Overall, it's a cloudy day in London.
